# sample4 - Functional API

Sequential API では表現できない複雑なモデルを **Functional API** で構築します。  
複数入力・複数出力・残差接続（Residual Connection）などに対応できます。

| ステップ | 内容 |
|----------|------|
| 1 | Sequential API との違い |
| 2 | 基本的な Functional API |
| 3 | 複数入力モデル |
| 4 | 残差接続（ResNet 風） |

In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # GPU を無効化して CPU で実行
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
print("TensorFlow:", tf.__version__)

I0000 00:00:1771826905.906600  119386 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1771826906.850455  119386 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow: 2.22.0-dev20260222


## 1. Sequential API との違い

```python
# Sequential API: 層を順番に積むだけ
model = tf.keras.Sequential([layer1, layer2, layer3])

# Functional API: 入出力を明示的に接続する
x = inputs = tf.keras.Input(shape=(4,))
x = layer1(x)
x = layer2(x)
outputs = layer3(x)
model = tf.keras.Model(inputs, outputs)
```

## 2. 基本的な Functional API

In [2]:
# データ準備
iris = load_iris()
X, y = iris.data, iris.target
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Functional API でモデル定義
inputs = tf.keras.Input(shape=(4,))
x = tf.keras.layers.Dense(16, activation='relu')(inputs)
x = tf.keras.layers.Dense(8,  activation='relu')(x)
outputs = tf.keras.layers.Dense(3, activation='softmax')(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 4)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │            80 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            27 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243 (972.00 B)

 Trainable params: 243 (972.00 B)

 Non-trainable params: 0 (0.00 B)

In [3]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=0)

loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"テスト Accuracy: {acc:.4f}")

テスト Accuracy: 0.9333


## 3. 複数入力モデル

数値特徴量とカテゴリ特徴量を別々に処理して結合するモデルです。

In [4]:
# 数値入力（がく・花びらの長さ）
input_num = tf.keras.Input(shape=(2,), name='numerical')
x_num = tf.keras.layers.Dense(8, activation='relu')(input_num)

# カテゴリ入力（がく・花びらの幅）
input_cat = tf.keras.Input(shape=(2,), name='categorical')
x_cat = tf.keras.layers.Dense(8, activation='relu')(input_cat)

# 結合
merged = tf.keras.layers.Concatenate()([x_num, x_cat])
x = tf.keras.layers.Dense(8, activation='relu')(merged)
outputs = tf.keras.layers.Dense(3, activation='softmax')(x)

multi_model = tf.keras.Model(inputs=[input_num, input_cat], outputs=outputs)
multi_model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ numerical           │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ categorical         │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 8)         │         24 │ numerical[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 8)         │         24 │ categorical[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16)        │          0 │ dense_3[0][0],    │
│ (Concatenate)       │                   │            │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 8)         │        136 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 3)         │         27 │ dense_5[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 211 (844.00 B)

 Trainable params: 211 (844.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
# データを2つに分割して入力
X_num = X_train[:, :2]
X_cat = X_train[:, 2:]
X_num_test = X_test[:, :2]
X_cat_test = X_test[:, 2:]

multi_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
multi_model.fit([X_num, X_cat], y_train, epochs=100, batch_size=16, verbose=0)

loss, acc = multi_model.evaluate([X_num_test, X_cat_test], y_test, verbose=0)
print(f"複数入力モデル Accuracy: {acc:.4f}")

複数入力モデル Accuracy: 1.0000


## 4. 残差接続（ResNet 風）

入力をスキップして後の層に直接足し合わせます。深いネットワークの学習安定化に有効です。

In [6]:
inputs = tf.keras.Input(shape=(4,))
x = tf.keras.layers.Dense(16, activation='relu')(inputs)

# 残差ブロック
shortcut = x
x = tf.keras.layers.Dense(16, activation='relu')(x)
x = tf.keras.layers.Dense(16)(x)
x = tf.keras.layers.Add()([x, shortcut])  # 残差接続
x = tf.keras.layers.Activation('relu')(x)

outputs = tf.keras.layers.Dense(3, activation='softmax')(x)
res_model = tf.keras.Model(inputs=inputs, outputs=outputs)

res_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
res_model.fit(X_train, y_train, epochs=100, batch_size=16, verbose=0)

loss, acc = res_model.evaluate(X_test, y_test, verbose=0)
print(f"残差接続モデル Accuracy: {acc:.4f}")

残差接続モデル Accuracy: 1.0000
